# P39 - Metric Traveling Salesperson Problem

## Project 1: Experimental Results Analysis

**Student:** Muhammad Abdullah Khan Akazai  
**Student ID:** 138367  
**Program:** Master in Computational Engineering

This notebook analyzes the performance of Exhaustive Search and the Greedy Heuristic using the benchmark results.

## 1. Load the experimental data

The dataset contains results for both algorithms across different graph sizes and target densities.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

if Path.cwd().name == 'notebooks':
    results_path = Path('../data/results.csv')
else:
    results_path = Path('data/results.csv')

df = pd.read_csv(results_path)

numeric_columns = [
    'N', 'density', 'basic_ops', 'execution_time_sec',
    'tested_configs', 'solution_quality', 'optimality_gap_pct'
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Total records:', len(df))
print('Unique graph configurations:', df[['N', 'density']].drop_duplicates().shape[0])
display(df.head())

## 2. Solver status summary

This section counts successful optimal results, feasible greedy tours, missing tours and timeouts.

In [ ]:
status_summary = df.groupby(['solver_type', 'status']).size().unstack(fill_value=0)
display(status_summary)

status_summary.plot(kind='bar', figsize=(9, 5))
plt.title('Solver Status Summary')
plt.xlabel('Solver')
plt.ylabel('Number of Instances')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Execution time comparison

Execution times are grouped by graph size and solver. The vertical axis uses a logarithmic scale because the algorithms have substantially different running times. Exhaustive runs that reached the timeout are shown at approximately 90 seconds.

In [ ]:
runtime_summary = df.groupby(['solver_type', 'N'])['execution_time_sec'].median().unstack(0)
display(runtime_summary)

for solver in runtime_summary.columns:
    plt.plot(runtime_summary.index, runtime_summary[solver], marker='o', label=solver)

plt.yscale('log')
plt.title('Execution Time vs Number of Vertices')
plt.xlabel('Number of Vertices (N)')
plt.ylabel('Median Execution Time (seconds, log scale)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Basic operations comparison

The following chart compares the operation counts recorded by each implementation. The operation counters have algorithm-specific definitions, so they should not be interpreted as perfectly equivalent units of work.

In [ ]:
operations_summary = df.groupby(['solver_type', 'N'])['basic_ops'].median().unstack(0)
display(operations_summary)

for solver in operations_summary.columns:
    plt.plot(operations_summary.index, operations_summary[solver], marker='o', label=solver)

plt.yscale('log')
plt.title('Basic Operations vs Number of Vertices')
plt.xlabel('Number of Vertices (N)')
plt.ylabel('Median Basic Operations (log scale)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Greedy optimality gap

The optimality gap is available only when Exhaustive Search completed optimally and the Greedy algorithm found a feasible tour. Instances without a valid comparison are excluded.

In [ ]:
gap_data = df[
    (df['solver_type'] == 'greedy') &
    (df['optimality_gap_pct'].notna())
].copy()

display(gap_data[['N', 'density', 'solution_quality', 'optimality_gap_pct', 'status']])

if not gap_data.empty:
    scatter = plt.scatter(
        gap_data['N'],
        gap_data['optimality_gap_pct'],
        c=gap_data['density'],
        cmap='viridis',
        s=80
    )
    plt.colorbar(scatter, label='Target Density (%)')
    plt.title('Greedy Optimality Gap')
    plt.xlabel('Number of Vertices (N)')
    plt.ylabel('Optimality Gap (%)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print('No instances are available for optimality-gap comparison.')

## 6. Experimental limitations and interpretation

- Exhaustive Search has factorial worst-case time complexity and becomes impractical as the graph size increases.
- The Greedy Heuristic is much faster in these experiments but does not guarantee an optimal tour.
- A Greedy `no_tour` result means the heuristic failed to construct a tour; it does not prove that no Hamiltonian cycle exists.
- An Exhaustive `timeout` result does not establish whether a Hamiltonian cycle exists or what its optimal cost would be.
- Optimality gaps are reported only when both a completed exact solution and a feasible greedy solution are available.
- The generated graphs are sparse random subgraphs with Euclidean edge weights. They are not necessarily complete metric graphs, so this distinction must be considered when interpreting the experiments.
- Operation counters use implementation-specific definitions and should be interpreted with care.

## 7. Conclusion

The experiments illustrate the trade-off between exact optimization and computational efficiency. Exhaustive Search provides optimality when it completes, while the Greedy Heuristic offers much lower execution times but may fail to find a tour or return a non-optimal solution.